In [5]:
import pandas as pd
import numpy as np
from functools import reduce

In [6]:
# Reading files and dropping empty rows (for some reason there are some rows with only date)

# Denoted in USD
apple_ohlcv = pd.read_csv("AAPL.csv").dropna()
msci_emerging_ohlcv = pd.read_csv("MME=F.csv").dropna()
btc_ohlcv = pd.read_csv("BTC-USD.csv").dropna()
# Denoted in Euro
airbus_ohlcv = pd.read_csv("AIR.PA.csv").dropna()
lvmh_ohlcv = pd.read_csv("MC.PA.csv").dropna()
euro_stoxx_50_ohlcv = pd.read_csv("SC0D.DE.csv").dropna()
# IR and FX
euribor_3M = pd.read_csv("ECB Data Portal Euribor 3M.csv").dropna()
eur_usd = pd.read_csv("EURUSD=X.csv").dropna()

In [7]:
# Calculating Log Returns and Setting Proper dates

# US Denominated
apple_ohlcv['returns_aapl'] = np.log(apple_ohlcv['Adj Close'] / apple_ohlcv['Adj Close'].shift(1))
apple_ohlcv["Date"] = pd.to_datetime(apple_ohlcv["Date"], format='%Y-%m-%d')
msci_emerging_ohlcv['returns_msci'] = np.log(msci_emerging_ohlcv['Adj Close'] / msci_emerging_ohlcv['Adj Close'].shift(1))
msci_emerging_ohlcv["Date"] = pd.to_datetime(msci_emerging_ohlcv["Date"], format='%Y-%m-%d')
btc_ohlcv['returns_btc'] = np.log(btc_ohlcv['Adj Close'] / btc_ohlcv['Adj Close'].shift(1))
btc_ohlcv["Date"] = pd.to_datetime(btc_ohlcv["Date"], format='%Y-%m-%d')
# EUR Denominated
airbus_ohlcv['returns_airbus'] = np.log(airbus_ohlcv['Adj Close'] / airbus_ohlcv['Adj Close'].shift(1))
airbus_ohlcv["Date"] = pd.to_datetime(airbus_ohlcv["Date"], format='%Y-%m-%d')
lvmh_ohlcv['returns_lvmh'] = np.log(lvmh_ohlcv['Adj Close'] / lvmh_ohlcv['Adj Close'].shift(1))
lvmh_ohlcv["Date"] = pd.to_datetime(lvmh_ohlcv["Date"], format='%Y-%m-%d')
euro_stoxx_50_ohlcv['returns_eu50'] = np.log(euro_stoxx_50_ohlcv['Adj Close'] / euro_stoxx_50_ohlcv['Adj Close'].shift(1))
euro_stoxx_50_ohlcv["Date"] = pd.to_datetime(euro_stoxx_50_ohlcv["Date"], format='%Y-%m-%d')
# FX Rates
eur_usd["Date"] = pd.to_datetime(eur_usd["Date"], format='%Y-%m-%d')
eur_usd = eur_usd.rename(columns={"Adj Close": "EUR_USD"})   # Set Correct Name

# Combining Returns into One Portfolio
portfolio = [eur_usd[["Date", "EUR_USD"]], apple_ohlcv[['Date', 'returns_aapl']], msci_emerging_ohlcv[['Date', 'returns_msci']], btc_ohlcv[['Date', 'returns_btc']], airbus_ohlcv[['Date', 'returns_airbus']], lvmh_ohlcv[['Date', 'returns_lvmh']], euro_stoxx_50_ohlcv[['Date', 'returns_eu50']]]
portfolio = reduce(lambda left, right: pd.merge(left, right, on='Date', how='outer'), portfolio)    # Merge all DataFrames on the 'Date' column
portfolio = portfolio.sort_values(by='Date', ignore_index=True)     # Sorting based on date
portfolio = portfolio.fillna(0)                     # Setting non traded dates to have 0 return 
portfolio = portfolio.drop(portfolio.head(1).index) # Zeroth day has no returns
portfolio = portfolio.drop(portfolio.tail(1).index) # We do not need a first day of a new year
# We need to update EURUSD rates to be kept at the same as a previous close, not at zero
current_rate = 1.0
for i in range(portfolio.shape[0]):
    # Example operation: multiply 'A' by 2 and add to 'B', update in 'C'
    if portfolio.iloc[i, 1] != 0:
        current_rate = portfolio.iloc[i, 1]
    else:
        portfolio.iloc[i, 1] = current_rate
portfolio.to_csv('_portfolio_returns.csv')       # Saving

In [8]:
portfolio

,Date,EUR_USD,returns_aapl,returns_msci,returns_btc,returns_airbus,returns_lvmh,returns_eu50
1,2015-01-01,1.209863,0.000000,0.000000,-0.018738,0.000000,0.000000,0.000000
2,2015-01-02,1.208941,-0.009558,-0.014620,0.002489,0.004223,-0.010643,0.000000
3,2015-01-03,1.208941,0.000000,0.000000,-0.114028,0.000000,0.000000,0.000000
4,2015-01-04,1.208941,0.000000,0.000000,-0.061959,0.000000,0.000000,0.000000
5,2015-01-05,1.194643,-0.028576,-0.017637,0.038169,0.003006,-0.029471,-0.037808
...,...,...,...,...,...,...,...,...
3283,2023-12-27,1.104301,0.000518,0.005965,0.021462,-0.000143,-0.003123,0.002195
3284,2023-12-28,1.110864,0.002224,0.006899,-0.018938,0.001863,-0.006001,-0.003295
3285,2023-12-29,1.106819,-0.005439,0.000968,-0.012474,0.000716,0.003550,0.003295
3286,2023-12-30,1.106819,0.000000,0.000000,0.001365,0.000000,0.000000,0.000000


In [9]:
# Cleaning IR Data for Credit

# Filter the DataFrame to keep only dates within the period of interest
euribor_3M["DATE"] = pd.to_datetime(euribor_3M["DATE"], format='%Y-%m-%d')
euribor_3M = euribor_3M[(euribor_3M['DATE'] >= pd.to_datetime('2015-01-01')) & (euribor_3M['DATE'] <= pd.to_datetime('2024-01-01'))]
# Keeping only months for the Interest Rates Credit Dataset
euribor_3M['DATE'] = euribor_3M['DATE'].dt.to_period('M')   # Keep only month and year
euribor_3M = euribor_3M.drop(columns=['TIME PERIOD'])       # Drop not used columns
euribor_3M = euribor_3M.rename(columns={"Euribor 3-month - Historical close, average of observations through period (FM.M.U2.EUR.RT.MM.EURIBOR3MD_.HSTA)": "EURIBOR_3M"})   # Simplify Names
euribor_3M = euribor_3M.reset_index(drop=True)              # Update Indexes
euribor_3M.to_csv('_EURIBOR_monthly_rates.csv')

In [10]:
euribor_3M

,DATE,EURIBOR_3M
0,2015-01,0.0627
1,2015-02,0.0482
2,2015-03,0.0272
3,2015-04,0.0047
4,2015-05,-0.0104
...,...,...
103,2023-08,3.7803
104,2023-09,3.8800
105,2023-10,3.9676
106,2023-11,3.9716
